# Pipeline Filogenético Público — 18S Adeleorina v3.5
### Fluxo reprodutível com sequências públicas do GenBank

**Entrada principal:** arquivo FASTA contendo apenas sequências públicas.

**Objetivo:** demonstrar um pipeline completo para curadoria, alinhamento, trimming, inferência filogenética por Máxima Verossimilhança e Inferência Bayesiana, re-enraizamento e cálculo de p-distance.

> Esta versão foi preparada para publicação em repositório público e utiliza exclusivamente dados disponíveis publicamente.

**Fluxo geral:**
```
0. Preparar ambiente e diretórios
1. Curar e complementar o conjunto público do GenBank
2. Alinhar com MAFFT
3. Fazer trimming com trimAl
4. Inferir árvore ML com IQ-TREE
5. Inferir árvore Bayesiana com MrBayes
6. Re-enraizar com outgroup público
7. Calcular p-distance
8. Gerar arquivos de controle e resultados
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## ETAPA 0 — Setup
> Execute uma única vez por sessão do Colab.

### 0.1 — Criar pastas do projeto

In [ ]:
%%bash
mkdir -p project/{raw,metadata,aln,tree,dist,logs}
echo "Estrutura criada:"
find project -maxdepth 1 -type d | sort


### 0.2 — Instalar dependências Python

In [ ]:
%%bash
pip -q install biopython
python3 -c "from Bio import SeqIO, AlignIO; print('✔ Biopython OK')"

### 0.3 — Instalar seqkit

In [ ]:
%%bash
set -e
wget -q https://github.com/shenwei356/seqkit/releases/download/v2.8.2/seqkit_linux_amd64.tar.gz
tar -xzf seqkit_linux_amd64.tar.gz
sudo mv seqkit /usr/local/bin/seqkit
rm seqkit_linux_amd64.tar.gz
echo "✔ seqkit $(seqkit version)"

### 0.4 — Instalar micromamba

In [ ]:
%%bash
set -e
cd /content
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH
echo "✔ micromamba $(micromamba --version)"

### 0.5 — Instalar MAFFT + trimAl + IQ-TREE3 + MrBayes + gotree
> Todas as ferramentas filogenéticas ficam no ambiente `bio`.


In [ ]:
%%bash
set -e
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

micromamba create -y -n bio -c conda-forge -c bioconda \
  mafft trimal iqtree mrbayes gotree

echo
echo "✔ Versões instaladas:"
micromamba run -n bio mafft   --version 2>&1 | head -1
micromamba run -n bio trimal  --version 2>&1 | head -1
micromamba run -n bio iqtree  --version 2>&1 | head -1
micromamba run -n bio mb      --version 2>&1 | head -1
micromamba run -n bio gotree  version


---
## ETAPA 1 — Preparação das sequências e novo alinhamento


### 1.1 — Upload do alinhamento anterior
> Envie o FASTA alinhado usado na análise anterior. Ele será convertido novamente em sequências não alinhadas antes da inclusão dos novos acessos.


In [ ]:
from google.colab import files
uploaded = files.upload()

# Detecta os FASTA enviados nesta célula.
fasta_uploads = [name for name in uploaded if name.lower().endswith((".fa", ".fas", ".fasta", ".fna"))]
if len(fasta_uploads) != 1:
    raise ValueError(f"Envie exatamente um FASTA. Detectados: {fasta_uploads}")

BASE_FILE = fasta_uploads[0]
print(f"✔ Arquivo-base: {BASE_FILE}")


In [ ]:
from pathlib import Path
import shutil

src = Path(BASE_FILE)
dst = Path("project/raw/base_aligned_previous.fasta")
shutil.copy2(src, dst)
print(f"✔ Cópia salva em: {dst}")


### 1.2 — Remover gaps e recuperar as sequências não alinhadas
> Esta etapa evita adicionar sequências novas diretamente a um alinhamento já processado. Todos os táxons serão realinhados juntos pelo MAFFT.


In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import re

entrada = "project/raw/base_aligned_previous.fasta"
saida = "project/raw/base_unaligned.fasta"

# IDs públicos que eventualmente devam ser excluídos antes do novo alinhamento.
# Deixe o conjunto vazio para manter todas as sequências.
EXCLUIR_IDS = set()

records = list(SeqIO.parse(entrada, "fasta"))

if not records:
    raise ValueError("O FASTA-base está vazio ou não pôde ser lido.")

ids = [r.id for r in records]
duplicados = sorted({sequence_id for sequence_id in ids if ids.count(sequence_id) > 1})

if duplicados:
    raise ValueError(f"IDs duplicados no arquivo-base: {duplicados}")

clean = []
removidas = []

for r in records:
    id_sem_versao = r.id.split(".")[0]

    if id_sem_versao in EXCLUIR_IDS:
        removidas.append(r.id)
        print(f"Sequência pública removida: {r.id}")
        continue

    seq = re.sub(r"[-.\s]", "", str(r.seq)).upper()

    if not seq:
        raise ValueError(f"Sequência vazia após remoção dos gaps: {r.id}")

    clean.append(
        SeqRecord(
            Seq(seq),
            id=r.id,
            description=""
        )
    )

SeqIO.write(clean, saida, "fasta")

print(f"Sequências de entrada: {len(records)}")
print(f"Sequências removidas: {len(removidas)}")
print(f"Sequências exportadas: {len(clean)}")
print(f"Arquivo gerado: {saida}")


### 1.3 — Definir acessos candidatos e representantes
> Os seis acessos serão baixados para conferência. Por padrão, entram na análise `PX921644`, `PX921641` e `PX921640`.
>
> Caso Amanda prefira outro isolado equivalente, altere somente `REP_SP2` ou `REP_AMEIVAE`.


In [ ]:
# E-mail exigido pelo NCBI Entrez. Substitua pelo seu e-mail institucional.
ENTREZ_EMAIL = "seuemail@email.com"

# Candidatos informados por Amanda Picelli
SEMITEANIATI = ["PX921644"]
SP2_CANDIDATOS = ["PX921641", "PX921642", "PX921643"]
AMEIVAE_CANDIDATOS = ["PX921639", "PX921640"]

# Representantes escolhidos para a árvore e para p-distance
REP_SP2 = "PX921641"
REP_AMEIVAE = "PX921640"

REFERENCIAS_BLAST = [
    "PX606498",
    "PX606497",
]

SELECIONADOS = (
    SEMITEANIATI
    + [REP_SP2, REP_AMEIVAE]
    + REFERENCIAS_BLAST
)

TODOS_CANDIDATOS = (
    SEMITEANIATI
    + SP2_CANDIDATOS
    + AMEIVAE_CANDIDATOS
    + REFERENCIAS_BLAST
)


assert REP_SP2 in SP2_CANDIDATOS
assert REP_AMEIVAE in AMEIVAE_CANDIDATOS
assert len(SELECIONADOS) == len(set(SELECIONADOS))

print("Acessos selecionados:", ", ".join(SELECIONADOS))
print("Acessos baixados para conferência:", ", ".join(TODOS_CANDIDATOS))


### 1.4 — Baixar sequências e metadados do GenBank
> A célula usa NCBI Entrez, registra os dados originais em GenBank e gera um FASTA apenas com os representantes selecionados.


In [ ]:
from Bio import Entrez, SeqIO
from pathlib import Path
import pandas as pd
import time

if ENTREZ_EMAIL == "seuemail@gmail.com":
    print("⚠ Recomenda-se substituir ENTREZ_EMAIL pelo seu e-mail real antes de publicar/reutilizar o notebook.")
Entrez.email = ENTREZ_EMAIL
Entrez.tool = "Public_Adeleorina_phylogeny_pipeline"

Path("project/raw").mkdir(parents=True, exist_ok=True)
Path("project/metadata").mkdir(parents=True, exist_ok=True)

records = []
for tentativa in range(1, 4):
    try:
        with Entrez.efetch(db="nucleotide", id=TODOS_CANDIDATOS,
                           rettype="gb", retmode="text") as handle:
            records = list(SeqIO.parse(handle, "genbank"))
        break
    except Exception as exc:
        if tentativa == 3:
            raise RuntimeError(f"Falha no download do NCBI após 3 tentativas: {exc}")
        print(f"Tentativa {tentativa} falhou; repetindo...")
        time.sleep(3 * tentativa)

# O NCBI pode devolver a versão no ID (ex.: PX921644.1). Comparação sem versão.
def sem_versao(x):
    return x.split(".")[0]

by_acc = {sem_versao(r.id): r for r in records}
ausentes = [acc for acc in TODOS_CANDIDATOS if acc not in by_acc]
if ausentes:
    raise ValueError(f"Acessos não recuperados no GenBank: {ausentes}")

SeqIO.write(records, "project/raw/genbank_candidates.gb", "genbank")
SeqIO.write(records, "project/raw/genbank_candidates.fasta", "fasta")

selected_records = []
rows = []
for acc in TODOS_CANDIDATOS:
    r = by_acc[acc]
    organism = r.annotations.get("organism", "")
    molecule = r.annotations.get("molecule_type", "")
    selected = acc in SELECIONADOS
    rows.append({
        "accession_requested": acc,
        "accession_version": r.id,
        "organism": organism,
        "description": r.description,
        "length_bp": len(r.seq),
        "molecule_type": molecule,
        "selected_for_analysis": selected,
    })
    if selected:
        # ID curto e estável; descrição fica na tabela de metadados.
        r.id = acc
        r.name = acc
        r.description = ""
        selected_records.append(r)

SeqIO.write(selected_records, "project/raw/genbank_selected.fasta", "fasta")
pd.DataFrame(rows).to_csv("project/metadata/genbank_accessions.tsv", sep="\t", index=False)

print(f"✔ {len(records)} registros candidatos baixados")
print(f"✔ {len(selected_records)} registros selecionados para a análise")
print("✔ Metadados: project/metadata/genbank_accessions.tsv")


### 1.5 — Validar download e seleção  ✔


In [ ]:
import pandas as pd
from Bio import SeqIO

meta = pd.read_csv("project/metadata/genbank_accessions.tsv", sep="\t")
display(meta)

baixados = set(meta["accession_requested"])
selecionados_fasta = {r.id for r in SeqIO.parse("project/raw/genbank_selected.fasta", "fasta")}

assert set(TODOS_CANDIDATOS) == baixados, "Nem todos os candidatos foram registrados."
assert set(SELECIONADOS) == selecionados_fasta, "O FASTA selecionado não corresponde à seleção definida."
assert (meta["length_bp"] > 0).all(), "Há sequência com comprimento zero."

print("✔ Todos os candidatos foram recuperados")
print("✔ Representantes selecionados:", ", ".join(sorted(selecionados_fasta)))


### 1.6 — Combinar sequências antigas e novas
> IDs duplicados são bloqueados para evitar que o mesmo acesso apareça duas vezes na análise.


In [ ]:
from Bio import SeqIO
from pathlib import Path

base = list(SeqIO.parse("project/raw/base_unaligned.fasta", "fasta"))
novas = list(SeqIO.parse("project/raw/genbank_selected.fasta", "fasta"))

base_ids = {r.id.split(".")[0] for r in base}
novos_ids = {r.id.split(".")[0] for r in novas}
repetidos = sorted(base_ids & novos_ids)
if repetidos:
    raise ValueError(
        "Estes acessos já existem no FASTA-base e seriam duplicados: " + ", ".join(repetidos)
    )

combinadas = base + novas
SeqIO.write(combinadas, "project/raw/all_unaligned.fasta", "fasta")

print(f"Sequências antigas: {len(base)}")
print(f"Sequências novas  : {len(novas)}")
print(f"Total para MAFFT  : {len(combinadas)}")
print("✔ Salvo em: project/raw/all_unaligned.fasta")


### 1.7 — Refazer o alinhamento múltiplo com MAFFT
> `--auto` seleciona a estratégia conforme o número e o tamanho das sequências. O alinhamento é reconstruído para todos os táxons, sem reutilizar as colunas do alinhamento anterior.


In [ ]:
%%bash
set -e
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

micromamba run -n bio mafft \
  --auto \
  --thread -1 \
  project/raw/all_unaligned.fasta \
  > project/aln/aln_input.fasta \
  2> project/logs/mafft.log

echo "✔ Novo alinhamento: project/aln/aln_input.fasta"
echo "✔ Log MAFFT: project/logs/mafft.log"
seqkit stats project/aln/aln_input.fasta


### 1.8 — Validar o novo alinhamento  ✔


In [ ]:
%%bash
set -e

echo "--- CHECKLIST 1.8 ---"
seqkit stats project/aln/aln_input.fasta

echo
python3 - <<'PY'
from Bio import AlignIO

aln = AlignIO.read("project/aln/aln_input.fasta", "fasta")
lens = {len(r.seq) for r in aln}
ids = [r.id for r in aln]
validos = set("ACGTRYSWKMBDHVNacgtryswkmbdhvn-")

print(f"Sequências : {len(aln)}")
print(f"Colunas    : {aln.get_alignment_length()}")

if len(lens) != 1:
    raise ValueError(f"Comprimentos distintos no MSA: {lens}")
print("✔ Todas as sequências têm o mesmo comprimento")

if len(ids) != len(set(ids)):
    raise ValueError("Há IDs duplicados no alinhamento final")
print("✔ IDs únicos")

problemas = [(r.id, set(str(r.seq)) - validos) for r in aln if set(str(r.seq)) - validos]
if problemas:
    raise ValueError(f"Caracteres não-IUPAC: {problemas}")
print("✔ Apenas caracteres IUPAC válidos")

esperados = {"PX921644", "PX921641", "PX921640"}
ausentes = esperados - set(ids)
if ausentes:
    raise ValueError(f"Novos acessos ausentes no alinhamento: {sorted(ausentes)}")
print("✔ Novos acessos presentes:", ", ".join(sorted(esperados)))
PY


---
## ETAPA 2 — Trimming (trimAl)

### 2.1 — Rodar trimAl
> `-automated1` é o modo recomendado para filogenética — remove colunas com excesso
> de gaps preservando o sinal filogenético.

In [ ]:
%%bash
set -e
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

micromamba run -n bio trimal \
  -in  project/aln/aln_input.fasta \
  -out project/aln/aln_trimmed.fasta \
  -automated1

echo "✔ Alinhamento trimado: project/aln/aln_trimmed.fasta"

In [ ]:
%%bash
set -e
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

micromamba run -n bio trimal \
  -in project/aln/aln_input.fasta \
  -out project/aln/aln_trimmed.fasta \
  -gt 0.5 \
  -st 0.001

echo "✔ Alinhamento trimado: project/aln/aln_trimmed.fasta"


### 2.2 — Gerar relatório HTML do trimming
> O relatório permite documentar as estatísticas do trimming e aumenta a rastreabilidade do workflow.


In [ ]:
%%bash
set -e
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

micromamba run -n bio trimal \
  -in      project/aln/aln_input.fasta \
  -automated1 \
  -htmlout project/logs/trimal_report.html || true

ls -lh project/logs/trimal_report.html
echo "✔ Relatório salvo: project/logs/trimal_report.html"

### 2.3 — Validar alinhamento trimado  ✔

In [ ]:
%%bash
set -e

echo "--- CHECKLIST 2.3 ---"
echo "Antes do trimming:"
seqkit stats project/aln/aln_input.fasta
echo
echo "Após trimming:"
seqkit stats project/aln/aln_trimmed.fasta
echo

python3 - <<'PY'
from Bio import AlignIO

antes  = AlignIO.read("project/aln/aln_input.fasta",   "fasta")
depois = AlignIO.read("project/aln/aln_trimmed.fasta",  "fasta")

col_antes  = antes.get_alignment_length()
col_depois = depois.get_alignment_length()
retidas    = col_depois / col_antes * 100

print(f"Colunas antes  : {col_antes}")
print(f"Colunas depois : {col_depois}")
print(f"Retidas        : {retidas:.1f}%")

if retidas < 30:
    print("⚠ Menos de 30% retidas — avalie usar -gappyout ou -gt 0.9")
elif retidas < 50:
    print("⚠ Trimming agressivo — verifique o relatório HTML")
else:
    print("✔ Porcentagem de retenção adequada")
PY

---
## ETAPA 3 — Máxima Verossimilhança (IQ-TREE3)

### 3.1 — Rodar IQ-TREE3
> ModelFinder (BIC) + 1000 UFBoot + 1000 SH-aLRT.
> Os rótulos dos nós ficam no formato `SH-aLRT/UFBoot` (ex: `95.3/98`).

In [ ]:
%%bash
set -e
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

micromamba run -n bio iqtree \
  -s     project/aln/aln_trimmed.fasta \
  -m     MFP \
  -B     1000 \
  --alrt 1000 \
  -nt    AUTO \
  -pre   project/tree/ML_haemogregarina

echo
echo "✔ Arquivos ML gerados:"
ls -lh project/tree/ML_haemogregarina.*

In [ ]:
%%bash
set -e

export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

mkdir -p project/tree

micromamba run -n bio iqtree \
  -s project/aln/aln_trimmed.fasta \
  -m MFP \
  -B 1000 \
  --alrt 1000 \
  -nt AUTO \
  -pre project/tree/ML_haemogregarina_68seq

echo
echo "✔ Arquivos ML gerados:"
ls -lh project/tree/ML_haemogregarina_68seq.*

### 3.2 — Checar modelo selecionado e suportes  ✔

In [ ]:
%%bash
set -euo pipefail

PREFIX="project/tree/ML_haemogregarina_68seq"
IQTREE="${PREFIX}.iqtree"
TREEFILE="${PREFIX}.treefile"

echo "--- CHECKLIST 3.2 ---"
echo

# Verificar se a análise terminou e gerou os arquivos necessários
for FILE in "$IQTREE" "$TREEFILE"; do
    if [[ ! -s "$FILE" ]]; then
        echo "✘ Arquivo não encontrado ou vazio: $FILE"
        echo
        echo "Arquivos disponíveis em project/tree:"
        ls -lh project/tree/
        exit 1
    fi
done

echo "Modelo selecionado segundo o BIC:"
grep -m1 -E "Best-fit model|Best-fit substitution model" "$IQTREE" \
    || echo "⚠ Informação do modelo não localizada"

echo
echo "Log-likelihood:"
grep -m1 "Log-likelihood of the tree" "$IQTREE" \
    || echo "⚠ Log-likelihood não localizado"

echo
python3 - "$TREEFILE" <<'PY'
import re
import sys

treefile = sys.argv[1]

with open(treefile, encoding="utf-8") as handle:
    tree = handle.read()

# No IQ-TREE, com --alrt e -B, os rótulos são:
# SH-aLRT/UFBoot
pairs = re.findall(
    r'(?<=[),])(\d+(?:\.\d+)?)/(\d+(?:\.\d+)?)(?=:)',
    tree
)

if not pairs:
    print("⚠ Rótulos SH-aLRT/UFBoot não encontrados no treefile")
    sys.exit(0)

alrt = [float(a) for a, _ in pairs]
ufboot = [float(b) for _, b in pairs]

well_supported = sum(
    a >= 80 and b >= 95
    for a, b in zip(alrt, ufboot)
)

print(f"Nós internos com suporte: {len(pairs)}")
print(
    f"SH-aLRT: mínimo={min(alrt):.1f} | "
    f"média={sum(alrt)/len(alrt):.1f} | "
    f"máximo={max(alrt):.1f}"
)
print(
    f"UFBoot:  mínimo={min(ufboot):.1f} | "
    f"média={sum(ufboot)/len(ufboot):.1f} | "
    f"máximo={max(ufboot):.1f}"
)
print(
    "\nNós bem suportados "
    f"(SH-aLRT ≥ 80 e UFBoot ≥ 95): "
    f"{well_supported}/{len(pairs)}"
)
PY

---
## ETAPA 4 — Inferência Bayesiana (MrBayes)

### 4.1 — Preparar entrada (Nexus + bloco MrBayes)

In [ ]:
from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.Nexus import Nexus # Import Bio.Nexus

# ---- PARÂMETROS — ajuste se necessário ----
NGEN       = 2_000_000   # aumente para 5_000_000 se PSRF > 1.01
SAMPLEFREQ = 500
NCHAINS    = 4
BURNIN     = 0.25        # 25%
OUTGROUPS  = [
    "DQ096835.1",  # Adelina dimidiata
    "DQ096836.2",  # Adelina grylli
]
PREFIX     = "project/tree/BI_haemogregarina"
# -------------------------------------------

aln_input = AlignIO.read("project/aln/aln_trimmed.fasta", "fasta")

# Ensure aln_input is a MultipleSeqAlignment object
# AlignIO.read can sometimes return an iterator
if not isinstance(aln_input, MultipleSeqAlignment):
    aln_input = MultipleSeqAlignment(list(aln_input))

# Create a Nexus object and add sequences
nexus_obj = Nexus.Nexus()
for record in aln_input:
    nexus_obj.add_sequence(record.id, str(record.seq).replace(' ', '')) # Remove spaces if any

# Write the Nexus data block to a temporary file
with open("project/tree/aln_mrbayes_temp.nex", "w") as f:
    nexus_obj.write_nexus_data(f)

# Confirmar que as duas sequências do outgroup existem no alinhamento
ids = [r.id for r in aln_input]
ausentes_outgroup = [x for x in OUTGROUPS if x not in ids]
if ausentes_outgroup:
    print(f"⚠ Sequência(s) de Adelina ausente(s): {ausentes_outgroup}")
    print("IDs disponíveis (primeiros 20):")
    for i in ids[:20]:
        print(f"  {i}")
else:
    print("✔ As duas sequências de Adelina foram encontradas:")
    for x in OUTGROUPS:
        print(f"  {x}")

OUTGROUP_MB = " ".join(OUTGROUPS)

# MrBayes block to be appended
bloco = f"""
begin mrbayes;
  set autoclose=yes nowarn=yes;

  lset nst=6 rates=invgamma;

  outgroup {OUTGROUP_MB};

  mcmcp ngen={NGEN}
        printfreq=1000
        samplefreq={SAMPLEFREQ}
        nchains={NCHAINS}
        nruns=2
        filename={PREFIX};
  mcmc;

  sumt burninfrac={BURNIN} filename={PREFIX};
end;
"""

# Combine the generated Nexus data with the MrBayes block
with open("project/tree/aln_mrbayes_temp.nex") as f_temp:
    nexus_content = f_temp.read()

with open("project/tree/mrbayes_input.nex", "w") as f_final:
    f_final.write(nexus_content)
    f_final.write(bloco)

print("✔ Arquivo criado: project/tree/mrbayes_input.nex")
print(f"  ngen={NGEN:,} | nchains={NCHAINS} | burnin={int(BURNIN*100)}%")

print("\n--- Conteúdo do bloco MrBayes ---")
print(bloco)
print("\n--- Conteúdo completo do arquivo mrbayes_input.nex (primeiras 100 linhas) ---")
with open("project/tree/mrbayes_input.nex") as fh:
    for n, line in enumerate(fh, start=1):
        if n > 100:
            break
        print(line, end="")


### 4.2 — Rodar MrBayes

In [ ]:
from pathlib import Path

nexus_file = Path("project/tree/mrbayes_input.nex")
text = nexus_file.read_text()

text = text.replace(
    "outgroup DQ096835.1 DQ096836.2;",
    """taxset Adelina_outgroup = DQ096835.1 DQ096836.2;
    outgroup Adelina_outgroup;"""
)

nexus_file.write_text(text)



In [ ]:
!grep -n -A3 -B3 -i "outgroup" project/tree/mrbayes_input.nex

In [ ]:
from pathlib import Path

arquivo = Path("project/tree/mrbayes_input.nex")

for numero, linha in enumerate(arquivo.read_text().splitlines(), start=1):
    if "outgroup" in linha.lower() or "taxset" in linha.lower():
        print(f"{numero}: {linha}")

In [ ]:
from pathlib import Path
import re

arquivo = Path("project/tree/mrbayes_input.nex")
texto = arquivo.read_text()

# Remove a definição do taxset das Adelina
texto = re.sub(
    r"^\s*taxset\s+Adelina_outgroup\s*=.*?;\s*$",
    "",
    texto,
    flags=re.MULTILINE | re.IGNORECASE
)

# Substitui qualquer linha anterior de outgroup
texto = re.sub(
    r"^\s*outgroup\s+.*?;\s*$",
    "    outgroup DQ096835.1;",
    texto,
    flags=re.MULTILINE | re.IGNORECASE
)

arquivo.write_text(texto)

print("Outgroup corrigido para DQ096835.1.")

In [ ]:
from pathlib import Path

for numero, linha in enumerate(
    Path("project/tree/mrbayes_input.nex").read_text().splitlines(),
    start=1
):
    if "outgroup" in linha.lower() or "taxset" in linha.lower():
        print(f"{numero}: {linha}")

In [ ]:
%%bash
set -e

export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

micromamba run -n bio mb project/tree/mrbayes_input.nex

### 4.3 — Verificar convergência (PSRF)  ✔
> **Critérios:**
> - PSRF ≈ 1.000 (máximo aceitável: 1.01)
> - Desvio médio das frequências de split < 0.01
> - Se não convergiu → execute a célula 4.4

In [ ]:
%%bash

echo "--- CHECKLIST 4.3 ---"
echo
echo "Desvio médio das frequências de split (deve ser < 0.01):"
echo "(verifique o output da célula 4.2 — linha 'Average standard deviation')"
echo
echo "Arquivos gerados:"
for f in project/tree/BI_haemogregarina.*; do
  [ -f "$f" ] && echo "  ✔ $f ($(du -h $f | cut -f1))"
done

echo
# Checar presença da árvore de consenso
if [ -f "project/tree/BI_haemogregarina.con.tre" ]; then
  echo "✔ Árvore de consenso BI encontrada"
else
  echo "⚠ Árvore de consenso não encontrada — o MrBayes pode não ter concluído"
fi

### 4.4 — [SE NÃO CONVERGIU] Reexecutar com ngen maior
> Execute **somente** se PSRF > 1.01 ou desvio médio > 0.01 na etapa 4.3.

In [ ]:
from Bio import AlignIO

NGEN_NOVO = 5_000_000
OUTGROUPS = ["DQ096835.1", "DQ096836.2"]
OUTGROUP_MB = " ".join(OUTGROUPS)
PREFIX_5M = "project/tree/BI_haemogregarina_5M"

bloco_5M = f"""
begin mrbayes;
  set autoclose=yes nowarn=yes;
  lset nst=6 rates=invgamma;
  outgroup {OUTGROUP_MB};
  mcmcp ngen={NGEN_NOVO}
        printfreq=1000
        samplefreq=500
        nchains=4
        nruns=2
        filename={PREFIX_5M};
  mcmc;
  sumt burninfrac=0.25 filename={PREFIX_5M};
end;
"""

with open("project/tree/aln_mrbayes.nex") as f:
    nexus = f.read()
with open("project/tree/mrbayes_input_5M.nex", "w") as f:
    f.write(nexus + bloco_5M)

print(f"✔ Arquivo 5M criado. Execute a célula abaixo para rodar.")

In [ ]:
%%bash
set -e
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

micromamba run -n bio mb project/tree/mrbayes_input_5M.nex

echo
echo "✔ Arquivos BI 5M:"
ls -lh project/tree/BI_haemogregarina_5M.* 2>/dev/null || true

---
## ETAPA 5 — Re-enraizamento (outgroup: *Adelina* spp.)

### 5.1 — Definir o outgroup com as duas sequências de *Adelina*

As duas sequências são tratadas conjuntamente como grupo externo:
- `DQ096835.1` — *Adelina dimidiata*;
- `DQ096836.2` — *Adelina grylli*.


In [ ]:
%%bash

# IDs do outgroup conforme aparecem no treefile
printf 'DQ096835.1\nDQ096836.2\n' > project/tree/outgroup.txt

echo "Outgroup definido:"
cat project/tree/outgroup.txt
echo

echo "Verificando presença na árvore ML:"
while read ID; do
  if grep -q "$ID" project/tree/ML_haemogregarina.treefile 2>/dev/null; then
    echo "  ✔ $ID encontrado"
  else
    echo "  ✖ $ID NÃO encontrado — ajuste o ID em outgroup.txt"
    echo "  IDs disponíveis (primeiros 15):"
    python3 -c "
import re
s = open('project/tree/ML_haemogregarina.treefile').read()
ids = sorted(set(re.findall(r'([A-Za-z0-9_.]+):', s)))
for i in ids[:15]: print('    ' + i)
" 2>/dev/null || true
  fi
done < project/tree/outgroup.txt

### 5.2 — Re-enraizar árvore ML

In [ ]:
%%bash
set -e
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

micromamba run -n bio gotree reroot outgroup \
  -i project/tree/ML_haemogregarina.treefile \
  -o project/tree/ML_haemogregarina.rooted.treefile \
  -l project/tree/outgroup.txt

echo "✔ Árvore ML re-enraizada: project/tree/ML_haemogregarina.rooted.treefile"

### 5.3 — Re-enraizar árvore BI

In [ ]:
%%bash
set -e
export MAMBA_ROOT_PREFIX=/content/mamba
export PATH=/content/bin:$PATH

# Detecta automaticamente se deve usar a corrida 2M ou 5M
BI_2M="project/tree/BI_haemogregarina.con.tre"
BI_5M="project/tree/BI_haemogregarina_5M.con.tre"

if [ -f "$BI_2M" ]; then
  USE="$BI_2M"
elif [ -f "$BI_5M" ]; then
  USE="$BI_5M"
else
  echo "⚠ Árvore BI não encontrada — complete a etapa 4 primeiro."
  exit 0
fi

micromamba run -n bio gotree reroot outgroup \
  -i "$USE" \
  -o project/tree/BI_haemogregarina.rooted.tre \
  -l project/tree/outgroup.txt \
  --format nexus

echo "✔ Árvore BI re-enraizada: project/tree/BI_haemogregarina.rooted.tre"
echo "  (origem: $USE)"

### 5.4 — Verificar raiz  ✔

In [ ]:
%%bash

echo "--- CHECKLIST 5.4 ---"

for TREE in \
  project/tree/ML_haemogregarina.rooted.treefile \
  project/tree/BI_haemogregarina.rooted.tre; do

  if [ ! -f "$TREE" ]; then
    echo "⚠ Não encontrado: $TREE"
    continue
  fi

  echo
  echo "Arquivo: $TREE"
  python3 - "$TREE" <<'PY'
import sys, re
s = open(sys.argv[1]).read().strip()
if 'DQ096835' in s and 'DQ096836' in s:
    print("  ✔ As duas sequências de Adelina estão presentes; confirme o clado basal no FigTree")
else:
    print("  ⚠ Posição do outgroup incerta — confirme no FigTree")
print(f"  Início do Newick: {s[:120].strip()}...")
PY
done

---
## ETAPA 6 — p-distance (pairwise deletion)

### 6.1 — Calcular p-distance do ingroup
> As duas sequências de *Adelina* são mantidas no alinhamento e nas árvores, mas excluídas da matriz de p-distance.


In [ ]:
from Bio import SeqIO
import pandas as pd
import numpy as np
import math

OUTGROUP_IDS = {"DQ096835.1", "DQ096836.2"}

recs_all = list(SeqIO.parse("project/aln/aln_trimmed.fasta", "fasta"))
ids_all = {r.id for r in recs_all}

# As Adelina permanecem no alinhamento das árvores, mas não entram na p-distance
recs = [r for r in recs_all if r.id not in OUTGROUP_IDS]

found_outgroups = sorted(OUTGROUP_IDS & ids_all)
missing_outgroups = sorted(OUTGROUP_IDS - ids_all)
print(f"Outgroups excluídos da p-distance: {found_outgroups}")
if missing_outgroups:
    print(f"⚠ Outgroups não encontrados no alinhamento: {missing_outgroups}")

names = [r.id for r in recs]
seqs  = [str(r.seq).upper() for r in recs]
n     = len(seqs)
BAD   = {"-", "N", "?", "X"}

def pdist(a, b):
    valid = diff = 0
    for x, y in zip(a, b):
        if x in BAD or y in BAD:
            continue
        valid += 1
        if x != y:
            diff += 1
    return (diff / valid, valid) if valid else (math.nan, 0)

mat   = np.full((n, n), np.nan)
pairs = []
for i in range(n):
    mat[i, i] = 0.0
    for j in range(i + 1, n):
        d, v = pdist(seqs[i], seqs[j])
        mat[i, j] = mat[j, i] = d
        pairs.append((names[i], names[j], d, v))

df_pairs = pd.DataFrame(pairs, columns=["seq1", "seq2", "p_distance", "sites_compared"])
df_mat   = pd.DataFrame(mat, index=names, columns=names)

df_pairs.to_csv("project/dist/pdist_pairs.tsv", sep="\t", index=False)
df_mat.to_csv("project/dist/pdist_matrix.tsv", sep="\t", index=True)

print(f"✔ p-distance calculada para {n} sequências do ingroup")
print("✔ Arquivos salvos:")
print("  project/dist/pdist_pairs.tsv")
print("  project/dist/pdist_matrix.tsv")


In [ ]:
!pip -q install biopython pandas numpy openpyxl

from google.colab import files
from Bio import SeqIO
import pandas as pd
import numpy as np
import math
from pathlib import Path

# Upload do alinhamento
uploaded = files.upload()

fasta_files = [
    filename
    for filename in uploaded
    if filename.lower().endswith((".fasta", ".fas", ".fa", ".fna"))
]

if not fasta_files:
    raise FileNotFoundError(
        "Nenhum arquivo FASTA foi enviado."
    )

ALIGNMENT = Path(fasta_files[0])

OUTPUT_DIR = Path("project/dist")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Arquivo de alinhamento selecionado: {ALIGNMENT}")

In [ ]:
# ============================================================
# P-DISTANCE — CONJUNTO PÚBLICO COMPLETO
# Google Colab
#
# Método:
#   p-distance não corrigida
#   pairwise deletion
#
# As sequências do outgroup permanecem na filogenia, mas são
# excluídas das análises de p-distance.
# ============================================================

!pip -q install biopython pandas numpy openpyxl

from Bio import SeqIO
import pandas as pd
import numpy as np
import math
from pathlib import Path

ALIGNMENT = Path("project/aln/aln_trimmed.fasta")
OUTPUT_DIR = Path("project/dist")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Outgroups públicos
OUTGROUP_IDS = {
    "DQ096835.1",
    "DQ096836.2",
}

if not ALIGNMENT.exists():
    raise FileNotFoundError(f"Alinhamento não encontrado: {ALIGNMENT}")

records_all = list(SeqIO.parse(str(ALIGNMENT), "fasta"))
if not records_all:
    raise ValueError("Nenhuma sequência foi encontrada no alinhamento.")

records = [r for r in records_all if r.id not in OUTGROUP_IDS]

names = [r.id for r in records]
seqs = [str(r.seq).upper() for r in records]

if len(records) < 2:
    raise ValueError("São necessárias pelo menos duas sequências do ingroup.")

VALID = {"A", "C", "G", "T"}

def calculate_pdistance(a, b):
    valid_sites = 0
    differences = 0

    for x, y in zip(a, b):
        if x not in VALID or y not in VALID:
            continue
        valid_sites += 1
        if x != y:
            differences += 1

    if valid_sites == 0:
        return math.nan, 0, 0

    return differences / valid_sites, valid_sites, differences

n = len(seqs)
matrix = np.full((n, n), np.nan)
pairs = []

for i in range(n):
    matrix[i, i] = 0.0

    for j in range(i + 1, n):
        d, sites, differences = calculate_pdistance(seqs[i], seqs[j])
        matrix[i, j] = matrix[j, i] = d

        pairs.append({
            "seq1": names[i],
            "seq2": names[j],
            "differences": differences,
            "sites_compared": sites,
            "p_distance": d,
            "p_distance_percent": d * 100 if not math.isnan(d) else math.nan,
        })

df_pairs_full = pd.DataFrame(pairs)
df_matrix_full = pd.DataFrame(matrix, index=names, columns=names)
df_matrix_full.index.name = "sequence"
df_matrix_full_percent = df_matrix_full * 100

df_pairs_full.to_csv(
    OUTPUT_DIR / "pdist_pairs_full.tsv",
    sep="\t",
    index=False,
    na_rep="NA"
)

df_matrix_full.to_csv(
    OUTPUT_DIR / "pdist_matrix_full.tsv",
    sep="\t",
    index=True,
    na_rep="NA"
)

df_matrix_full_percent.to_csv(
    OUTPUT_DIR / "pdist_matrix_full_percent.tsv",
    sep="\t",
    index=True,
    na_rep="NA"
)

print(f"Sequências do ingroup: {n}")
print(f"Pares calculados: {len(df_pairs_full)}")
print("Arquivos salvos em project/dist/")


In [ ]:
from google.colab import files
import pandas as pd

excel_output = OUTPUT_DIR / "pdistance_results_public.xlsx"

with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:
    df_matrix_full.to_excel(
        writer,
        sheet_name="Matrix proportion"
    )

    df_matrix_full_percent.to_excel(
        writer,
        sheet_name="Matrix percent"
    )

    df_pairs_full.to_excel(
        writer,
        sheet_name="Unique pairs",
        index=False
    )

print(f"Arquivo Excel salvo em: {excel_output}")
files.download(str(excel_output))


In [ ]:
from google.colab import files
import shutil

# Compacta toda a pasta de resultados
shutil.make_archive(
    "pdistance_results",
    "zip",
    "project/dist"
)

# Baixa o arquivo ZIP
files.download("pdistance_results.zip")

### 6.2 — Resumo estatístico  ✔

In [ ]:
upper = mat[np.triu_indices(n, 1)]

print("--- CHECKLIST 6.2 ---")
print(f"Sequências do ingroup: {n}")
print(f"Pares       : {len(df_pairs)}")
print(f"Colunas     : {len(seqs[0])}")
print()
print("p-distance (pairwise deletion):")
print(f"  Mínima  = {np.nanmin(upper):.4f}")
print(f"  Média   = {np.nanmean(upper):.4f}")
print(f"  Mediana = {np.nanmedian(upper):.4f}")
print(f"  Máxima  = {np.nanmax(upper):.4f}")
print()
print("Sítios comparados por par:")
print(f"  Mínimo  = {df_pairs.sites_compared.min():.0f}")
print(f"  Mediana = {df_pairs.sites_compared.median():.0f}")
print(f"  Máximo  = {df_pairs.sites_compared.max():.0f}")
print()

identicos = df_pairs[df_pairs.p_distance == 0]
if len(identicos) > 0:
    print(f"⚠ {len(identicos)} par(es) com p-distance = 0 (sequências idênticas):")
    for _, row in identicos.iterrows():
        print(f"   {row.seq1}  ↔  {row.seq2}")
else:
    print("✔ Nenhum par com p-distance = 0")

### 6.3 — [OPCIONAL] p-distance intragrupo
> Edite a lista `GRUPO_IDS` com os IDs do seu grupo de interesse.

In [ ]:
from Bio import SeqIO
import pandas as pd
import numpy as np
import math

# IDs públicos do grupo de interesse.
# Edite esta lista utilizando apenas acessos disponíveis publicamente.
GRUPO_IDS = [
    "OQ388265.1",
    "OQ377133.1",
    "OQ377233.1",
    "OQ377555.1",
    "OQ377557.1",
    "OQ377565.1",
    "OQ377710.1",
    "HQ224957.1",
    "MN879392.1",
    "PX921644",
    "PX921641",
    "PX921640",
]

rec_map = {
    r.id: str(r.seq).upper()
    for r in SeqIO.parse("project/aln/aln_trimmed.fasta", "fasta")
}

ausentes = [x for x in GRUPO_IDS if x not in rec_map]
if ausentes:
    print(f"IDs públicos não encontrados: {ausentes}")

nomes = [x for x in GRUPO_IDS if x in rec_map]
seqs_g = [rec_map[x] for x in nomes]

if len(seqs_g) < 2:
    raise ValueError("Menos de duas sequências selecionadas foram encontradas.")

VALID = {"A", "C", "G", "T"}

def pdist(a, b):
    valid = 0
    diff = 0

    for x, y in zip(a, b):
        if x not in VALID or y not in VALID:
            continue
        valid += 1
        if x != y:
            diff += 1

    return (diff / valid, valid) if valid else (math.nan, 0)

ng = len(seqs_g)
matg = np.full((ng, ng), np.nan)
pairsg = []

for i in range(ng):
    matg[i, i] = 0.0

    for j in range(i + 1, ng):
        d, v = pdist(seqs_g[i], seqs_g[j])
        matg[i, j] = matg[j, i] = d
        pairsg.append((nomes[i], nomes[j], d, v))

df_g = pd.DataFrame(
    pairsg,
    columns=["seq1", "seq2", "p_distance", "sites_compared"]
)

df_g.to_csv(
    "project/dist/pdist_public_group_pairs.tsv",
    sep="\t",
    index=False
)

up = matg[np.triu_indices(ng, 1)]

print(f"Grupo público: {ng} sequências | {len(pairsg)} pares")
print("p-distance intragrupo:")
print(f"  Mínima = {np.nanmin(up):.4f}")
print(f"  Média  = {np.nanmean(up):.4f}")
print(f"  Máxima = {np.nanmax(up):.4f}")
print("Salvo: project/dist/pdist_public_group_pairs.tsv")


---
# ETAPA 6 - NOVA




p-distance das 10 sequências selecionadas

A p-distance não corrigida é calculada exclusivamente entre as dez sequências
selecionadas. Posições contendo gaps, dados ausentes ou nucleotídeos ambíguos
são excluídas separadamente em cada comparação (pairwise deletion).

In [ ]:
# ============================================================
# P-DISTANCE — SUBCONJUNTO DE SEQUÊNCIAS PÚBLICAS
#
# Utilize esta célula para produzir uma matriz reduzida com
# acessos públicos escolhidos para fins demonstrativos.
# ============================================================

from Bio import SeqIO
from pathlib import Path
import pandas as pd
import numpy as np
import math

ALIGNMENT = Path("project/aln/aln_trimmed.fasta")
OUTPUT_DIR = Path("project/dist/pdistance_selected")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Subconjunto composto exclusivamente por acessos públicos.
# A lista pode ser editada conforme o objetivo da demonstração.
SELECTED_ACCESSIONS = [
    "PX606498",
    "PX606497",
    "MH503892",
    "MW540605",
    "MW540606",
    "MF476203",
    "KX507248",
    "MN160460",
]

def accession_key(sequence_id):
    return sequence_id.split(".")[0]

if not ALIGNMENT.exists():
    raise FileNotFoundError(
        f"Alinhamento não encontrado: {ALIGNMENT}"
    )

records_all = list(SeqIO.parse(str(ALIGNMENT), "fasta"))

record_map = {}
for record in records_all:
    key = accession_key(record.id)
    if key in record_map:
        raise ValueError(
            f"Identificador duplicado após normalização: {key}"
        )
    record_map[key] = record

missing = [
    accession
    for accession in SELECTED_ACCESSIONS
    if accession not in record_map
]

if missing:
    raise ValueError(
        "Acessos públicos ausentes do alinhamento:\n"
        + "\n".join(f"  - {x}" for x in missing)
    )

selected_records = [
    record_map[x]
    for x in SELECTED_ACCESSIONS
]

lengths = {len(r.seq) for r in selected_records}
if len(lengths) != 1:
    raise ValueError(
        "As sequências selecionadas não possuem o mesmo "
        "comprimento de alinhamento."
    )

VALID = {"A", "C", "G", "T"}

def calculate_pdistance(a, b):
    valid_sites = 0
    differences = 0

    for x, y in zip(a, b):
        if x not in VALID or y not in VALID:
            continue
        valid_sites += 1
        if x != y:
            differences += 1

    if valid_sites == 0:
        return math.nan, 0, 0

    return differences / valid_sites, valid_sites, differences

n = len(SELECTED_ACCESSIONS)
distance_matrix = np.full((n, n), np.nan)
sites_matrix = np.zeros((n, n), dtype=int)
rows = []

seqs = {
    accession: str(record_map[accession].seq).upper()
    for accession in SELECTED_ACCESSIONS
}

for i, acc_i in enumerate(SELECTED_ACCESSIONS):
    distance_matrix[i, i] = 0.0

    for j in range(i + 1, n):
        acc_j = SELECTED_ACCESSIONS[j]

        d, sites, differences = calculate_pdistance(
            seqs[acc_i],
            seqs[acc_j]
        )

        distance_matrix[i, j] = distance_matrix[j, i] = d
        sites_matrix[i, j] = sites_matrix[j, i] = sites

        rows.append({
            "accession_1": acc_i,
            "accession_2": acc_j,
            "differences": differences,
            "sites_compared": sites,
            "p_distance": d,
            "p_distance_percent": d * 100 if not math.isnan(d) else math.nan,
        })

df_distance = pd.DataFrame(
    distance_matrix,
    index=SELECTED_ACCESSIONS,
    columns=SELECTED_ACCESSIONS
)

df_distance_percent = df_distance * 100

df_sites = pd.DataFrame(
    sites_matrix,
    index=SELECTED_ACCESSIONS,
    columns=SELECTED_ACCESSIONS
)

df_pairs = pd.DataFrame(rows)

df_distance.to_csv(
    OUTPUT_DIR / "pdist_selected_matrix_proportion.tsv",
    sep="\t",
    float_format="%.6f",
    na_rep="NA"
)

df_distance_percent.to_csv(
    OUTPUT_DIR / "pdist_selected_matrix_percent.tsv",
    sep="\t",
    float_format="%.2f",
    na_rep="NA"
)

df_sites.to_csv(
    OUTPUT_DIR / "pdist_selected_sites_compared.tsv",
    sep="\t"
)

df_pairs.to_csv(
    OUTPUT_DIR / "pdist_selected_unique_pairs.tsv",
    sep="\t",
    index=False,
    float_format="%.6f",
    na_rep="NA"
)

SeqIO.write(
    selected_records,
    OUTPUT_DIR / "selected_public_sequences.fasta",
    "fasta"
)

excel_output = OUTPUT_DIR / "p_distance_selected_public_sequences.xlsx"

with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:
    df_distance_percent.to_excel(writer, sheet_name="Percent")
    df_distance.to_excel(writer, sheet_name="Proportion")
    df_sites.to_excel(writer, sheet_name="Sites compared")
    df_pairs.to_excel(writer, sheet_name="Unique pairs", index=False)

print("=" * 60)
print("P-DISTANCE — PUBLIC DATASET")
print("=" * 60)
print(f"Sequências selecionadas: {n}")
print(f"Pares únicos: {len(df_pairs)}")
print(f"Resultados: {OUTPUT_DIR}")


In [ ]:
from google.colab import files
import shutil

zip_output = shutil.make_archive(
    "p_distance_10_sequences",
    "zip",
    root_dir=OUTPUT_DIR
)

print(f"✔ Arquivo compactado: {zip_output}")

files.download(zip_output)



---



---
## ETAPA 7 — Exportar resultados

### 7.1 — Listar arquivos finais  ✔

In [ ]:
%%bash

echo "=== CHECKLIST FINAL ==="
echo

check() {
  if [ -f "$1" ]; then
    SIZE=$(du -h "$1" | cut -f1)
    echo "  ✔ $1 ($SIZE)"
  else
    echo "  ✖ AUSENTE: $1"
  fi
}

echo "--- Sequências e metadados ---"
check project/raw/base_unaligned.fasta
check project/raw/genbank_candidates.gb
check project/raw/genbank_selected.fasta
check project/raw/all_unaligned.fasta
check project/metadata/genbank_accessions.tsv

echo
echo "--- Alinhamentos ---"
check project/aln/aln_input.fasta
check project/aln/aln_trimmed.fasta

echo
echo "--- Árvores ML ---"
check project/tree/ML_haemogregarina.treefile
check project/tree/ML_haemogregarina.rooted.treefile
check project/tree/ML_haemogregarina.iqtree

echo
echo "--- Árvores BI ---"
check project/tree/BI_haemogregarina.con.tre
check project/tree/BI_haemogregarina.rooted.tre

echo
echo "--- Distâncias ---"
check project/dist/pdist_pairs.tsv
check project/dist/pdist_matrix.tsv

echo
echo "--- Logs / Suplementar ---"
check project/logs/mafft.log
check project/logs/trimal_report.html


### 7.2 — Compactar e baixar resultados

In [ ]:
%%bash
set -e

tar -czf pipeline_results.tar.gz project/
ls -lh pipeline_results.tar.gz
echo
echo "✔ Pronto. Execute a célula abaixo para baixar."

In [ ]:
from google.colab import files
files.download("pipeline_results.tar.gz")

---
## Referência rápida — principais arquivos do workflow

| Arquivo | Uso |
|---|---|
| `genbank_accessions.tsv` | Rastreabilidade dos acessos públicos utilizados |
| `mafft.log` | Registro do alinhamento múltiplo |
| `ML_haemogregarina.rooted.treefile` | Árvore ML re-enraizada |
| `BI_haemogregarina.rooted.tre` | Árvore Bayesiana re-enraizada |
| `ML_haemogregarina.iqtree` | Modelo selecionado e estatísticas do IQ-TREE |
| `trimal_report.html` | Estatísticas do trimming |
| `pdist_matrix_full.tsv` | Matriz completa de p-distance |
| `pdist_public_group_pairs.tsv` | Distâncias do subconjunto público selecionado |

**Abrindo no FigTree:**
- ML: `Node Labels → label` exibe `SH-aLRT/UFBoot`;
- BI: `Node Labels → prob` exibe probabilidades posteriores.
